In [1]:
import os 
from dotenv import load_dotenv
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_API_KEY

'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'

In [9]:
from langchain_groq import ChatGroq
model = ChatGroq(model="llama-3.1-8b-instant",api_key=GROQ_API_KEY)
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002198F6FC9D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002198F6FE780>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [51]:
from langchain_core.messages import HumanMessage,AIMessage

response = model.invoke([HumanMessage(content="What is the capital of France?")])

In [12]:
response 

AIMessage(content='The capital of France is Paris.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 42, 'total_tokens': 50, 'completion_time': 0.006079791, 'completion_tokens_details': None, 'prompt_time': 0.003663418, 'prompt_tokens_details': None, 'queue_time': 0.15883517, 'total_time': 0.009743209}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eaaf8-ebb9-7021-ad06-a78ad2703ddd-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 8, 'total_tokens': 50})

In [16]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory


store={}


def get_session_history(key:str)->BaseChatMessageHistory:
    if key not in store:
        store[key]=ChatMessageHistory()
    return store[key]

model_with_memory = RunnableWithMessageHistory(model,get_session_history)


In [17]:
config = {
    "configurable":{"session_id":"chat1"}
}

In [18]:
model_with_memory.invoke([HumanMessage(content="What is the capital of France?")],config=config)

AIMessage(content='The capital of France is Paris.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 42, 'total_tokens': 50, 'completion_time': 0.007016168, 'completion_tokens_details': None, 'prompt_time': 0.009944032, 'prompt_tokens_details': None, 'queue_time': 0.108208792, 'total_time': 0.0169602}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eab06-3f8b-7181-9317-d0de3ca40ea8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 8, 'total_tokens': 50})

In [19]:
model_with_memory.invoke([HumanMessage(content="What question did i ask?")],config=config)

AIMessage(content='You asked "What is the capital of France?"', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 65, 'total_tokens': 76, 'completion_time': 0.009478573, 'completion_tokens_details': None, 'prompt_time': 0.017004031, 'prompt_tokens_details': None, 'queue_time': 0.114188587, 'total_time': 0.026482604}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eab06-a93c-7312-8546-a85fff1d7892-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 65, 'output_tokens': 11, 'total_tokens': 76})

## Using Prompt template


In [20]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system","You are a helpful assistant that answers questions based on the conversation history."),
    MessagesPlaceholder(variable_name="messages")
])

In [24]:
chain = prompt | model

model_with_memory = RunnableWithMessageHistory(chain,get_session_history)

In [26]:
config2 = {"configurable":{"session_id":"chat2"}}

model_with_memory.invoke([HumanMessage(content="What is the capital of France?")],config=config2)
model_with_memory.invoke([HumanMessage(content="My Name is John Doe, I'm  a software engineer")],config=config2)

AIMessage(content="Nice to meet you, John Doe. You're a software engineer, that's a great field. What kind of projects have you been working on recently?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 114, 'total_tokens': 146, 'completion_time': 0.045075079, 'completion_tokens_details': None, 'prompt_time': 0.010448905, 'prompt_tokens_details': None, 'queue_time': 0.102066673, 'total_time': 0.055523984}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eab10-914a-7321-abf0-b61ac3ee0469-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 114, 'output_tokens': 32, 'total_tokens': 146})

In [28]:
model_with_memory.invoke([HumanMessage(content="What is my name and What do I do?")],config=config2)

AIMessage(content="Your name is John Doe, and you're a software engineer.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 165, 'total_tokens': 179, 'completion_time': 0.039779645, 'completion_tokens_details': None, 'prompt_time': 0.064530984, 'prompt_tokens_details': None, 'queue_time': 0.094789684, 'total_time': 0.104310629}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eab11-23be-7eb1-90aa-2f7dd5794e13-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 165, 'output_tokens': 14, 'total_tokens': 179})

In [44]:
### Adding in more complexity 
complex_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that answers questions to best of it's ability also in {language} language."),
    MessagesPlaceholder(variable_name="messages")
])

In [45]:
chain = complex_prompt | model


modelwithmemory = RunnableWithMessageHistory(chain,get_session_history,input_messages_key="messages")

In [46]:
config3 = {"configurable":{"session_id":"chat3"}}

In [47]:
modelwithmemory.invoke(
    {'messages': [HumanMessage(content="Hello my self is John Doe, I am a software engineer at Google and my age is 35.")],'language':"hindi"},
    config=config3
    )

AIMessage(content='नमस्ते जॉन डू, आप गूगल में सॉफ्टवेयर इंजीनियर हैं और आपकी उम्र 35 वर्ष है। आपका काम कैसा जा रहा है?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 75, 'total_tokens': 127, 'completion_time': 0.09891589, 'completion_tokens_details': None, 'prompt_time': 0.007698298, 'prompt_tokens_details': None, 'queue_time': 0.049935569, 'total_time': 0.106614188}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eab1b-4f3b-7983-9a01-ebff1928ca64-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 75, 'output_tokens': 52, 'total_tokens': 127})

In [49]:
modelwithmemory.invoke(
    {'messages': [HumanMessage(content="Whats My Name? and what is my age?")],'language':"English"},
    config=config3
    )

AIMessage(content='Your name is John Doe, and your age is 35 years old.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 226, 'total_tokens': 242, 'completion_time': 0.019984198, 'completion_tokens_details': None, 'prompt_time': 0.017332614, 'prompt_tokens_details': None, 'queue_time': 0.051855002, 'total_time': 0.037316812}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eab1c-1d49-7533-a2f7-fe0a125633bb-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 226, 'output_tokens': 16, 'total_tokens': 242})

In [55]:
!pip install transformers


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [57]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer=trim_messages(
    max_tokens=45,
    strategy="last",
    include_system=True,
    allow_partial=False,
    start_on="human"
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

TypeError: trim_messages() missing 1 required keyword-only argument: 'token_counter'